## ***SEC***

***Teams***
1. Alabama
2. Arkansas
3. Auburn
4. Florida
5. Georgia
6. Kentucky
7. Lousiana State
8. Mississippi (Ole Miss)
9. Mississippi State
10. Missouri
11. Oklahoma
12. South Carolina
13. Tennessee
14. Texas
15. Texas A&M
16. Vanderbilt

In [2]:
### Imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

***Scrapers***

In [1]:
'''
<div>, <h3>, <h4> BeautifulSoup Scraper
Teams:
- North Florida (Atlantic Sun)
- Davidson (Atlantic 10)
- Saint Louis (Atlantic 10)
- Big East: Creighton, UConn, DePaul, Marquette
- Big Sky: Montana State, Northern Colorado
- Big South: UNC Asheville
- Big West: Cal Poly, CSUN, Long Beach State
- Big 10: Indiana, Michigan, Michigan State, Minnesota, Northwestern, Ohio State, Washington
- Big 12: Baylor, Iowa State, Kansas, Kansas State, Oklahoma State, TCU, Utah, West Virginia
- CAA: Charleston, Drexel
- Horizon League: Purdue Fort Wayne
- Conference USA: Liberty, Middle Tennesse State
- Ivy League: Columbia, Dartmouth
- Metro Atlantic: Quinnipiac
- Mid-American: Akron, Ball State, Bowling Green, Buffalo, Central Michigan, Eastern Michigan, Kent State, Massachusetts, Miami (Ohio), Toledo
- Missouri Valley: Murray State, Northern Iowa
- Mountain West: Colorado State, Fresno State
- Ohio Valley: Tennessee Martin
- Patriot League: Bucknell
- SEC: Alabama, Florida, Georgia, Ole Miss, Mississippi State, Missouri, Oklahoma, Tennessee, Texas, Texas A&M
'''
def scrape_div_h3_h4(url, school, conference = "SEC"):
    # Make sure school can be fetched
    try:
        response = requests.get(url)
        response.raise_for_status()
    except Exception as e:
        print(f"Error fetching {school}: {e}")
        return pd.DataFrame()

    # Set up parser, variables
    soup = BeautifulSoup(response.text, "html.parser")
    staff_list = []
    current_department = None

    # Iterate through divs and h3s to find departments and staff cards
    for element in soup.find_all(["div", "h3"]):
        # If it's a department header
        if element.name == "h3" and "s-text-title" in element.get("class", []):
            current_department = element.get_text(strip=True)

        # If it's a staff card
        elif element.get("data-test-id") == "s-person-card-list__root":
            name_tag = element.find("h4")
            name = name_tag.get_text(strip=True) if name_tag else None

            title_tag = element.find("div", class_="s-person-details__position")
            title = title_tag.get_text(" ", strip=True) if title_tag else None

            if name:  # Skip blank cards
                staff_list.append({
                    "name": name,
                    "title": title,
                    "department": current_department,
                    "school": school,
                    "conference": conference
                })

    df = pd.DataFrame(staff_list)
    print(f"{school}: Scraped {len(df)} staff members.")
    time.sleep(2)  # gentle delay
    return df

In [ ]:
"""
Department headers are stored in a td class embedded in a tr class with the tag
heading "staff-directory-table-department__title".

Staff cards are stored in td classes embedded in tr classes with the tag heading
"staff-directory-table-position__name". The name is in an <a> tag, the title is 
in a <p> tag.

Schools:
- Big 10: Iowa, Nebraska, Penn State, Purdue
- Big 12: BYU, UCF, Cincinnati
- Mountain West: New Mexico, San Diego State, San Jose State
- SEC: Auburn
"""
def scrape_tr_td_directory_table(url, school, conference="SEC"):
    
    # Get the url, request and parse with error handling
    
    try:
        response = requests.get(url)
        response.raise_for_status()
    except Exception as e:
        print(f"Error fetching {school}: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(response.text, "html.parser")

    staff_list = []
    current_department = None

    # Traverse all <tr> elements in document order
    for row in soup.find_all("tr"):
        row_class = row.get("class", [])

        # --- Department header row ---
        if "staff-directory-table-department__head" in row_class:
            dept_cell = row.find("td", class_="staff-directory-table-department__title")
            if dept_cell:
                span = dept_cell.find("span")
                if span:
                    current_department = span.get_text(strip=True)
                else:
                    current_department = dept_cell.get_text(strip=True)

        # --- Staff member row ---
        elif "staff-directory-table-department__row" in row_class:
            cells = row.find_all("td")
            if len(cells) >= 2:
                # Name is usually in the first cell, inside <a>
                name_tag = cells[0].find("a", class_="staff-directory-table-member-position__link--name")
                name = name_tag.get_text(strip=True) if name_tag else cells[0].get_text(strip=True)

                # Title in the second cell
                title = cells[1].get_text(" ", strip=True) if len(cells) > 1 else ""

                # Append to list
                if name:
                    staff_list.append({
                        "name": name,
                        "title": title,
                        "department": current_department or "",
                        "school": school,
                        "conference": conference
                    })

    # Convert to DataFrame
    staff_df = pd.DataFrame(staff_list)
    print(f"{school}: Scraped {len(staff_df)} staff members.")
    time.sleep(1)
    return staff_df


In [ ]:
"""
Scrape staff directory pages where each department is stored in
<div data-department-id> sections, with <thead> for headers and <tbody> for rows.
The department name is in the first <th> of the <thead>.
Staff rows are in the <tbody> with names in the first <td> (inside <a>) and titles in the second <td>.

Schools:
- SEC: Arkansas
"""

def scrape_thead_tbody_tables(url, school, conference="SEC"):
    
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
    except Exception as e:
        print(f"Error fetching {school}: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(r.text, "html.parser")

    staff_list = []

    # Each department is wrapped in: <div data-department-id="...">
    dept_sections = soup.find_all("div", attrs={"data-department-id": True})

    for section in dept_sections:
        # Find the table in this department section
        table = section.find("table")
        if not table:
            continue

        # Extract department name from the first <th> in <thead>
        thead = table.find("thead")
        if not thead:
            continue

        first_th = thead.find("th")
        if not first_th:
            continue

        department = first_th.get_text(strip=True)

        # Now extract staff rows in <tbody>
        tbody = table.find("tbody")
        if not tbody:
            continue

        for row in tbody.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) < 2:
                continue

            # Name is inside <a> in the first column
            name_tag = cells[0].find("a")
            name = name_tag.get_text(strip=True) if name_tag else cells[0].get_text(strip=True)

            title = cells[1].get_text(strip=True) if len(cells) > 1 else ""

            # Clean em dashes, whitespace
            title = title.replace("—", "").strip()

            staff = {
                "school": school,
                "conference": conference,
                "department": department,
                "name": name,
                "title": title
            }

            staff_list.append(staff)

    df = pd.DataFrame(staff_list)
    print(f"{school}: Scraped {len(df)} staff members.")
    return pd.DataFrame(df)


In [27]:
"""
All staff and department elements are in <tr> rows. 
The department header is in a <tr> tag with a <td class='table-separator'>.
The staff members in that department follow until the next department header.
The staff cards are <tr class="staff-directory_table_row">. 
The name is nested in the first <td> in <a><div class="name">, the title is in the second <td>.

Schools:
- SEC: Kentucky, LSU, South Carolina
"""

def scrape_tr_tr_table_row(url, school, conference="SEC"):
    """
    Scrapes staff directory pages where department headers appear in
    <td class='table-separator'> and staff info appears in
    <tr class='staff-directory_table_row'> blocks.
    """

    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
    except Exception as e:
        print(f"Error fetching {school}: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(r.text, "html.parser")

    staff_list = []
    current_dept = None

    # Each department header
    for row in soup.find_all("tr"):
        # Department header
        dept_cell = row.find("td", class_="table-separator")
        if dept_cell:
            current_dept = dept_cell.get_text(strip=True)
            continue

        # Staff rows
        if "staff-directory_table_row" in row.get("class", []):
            cells = row.find_all("td")
            if len(cells) < 2:
                continue

            # Name is nested in <a><div class="name">
            name_tag = cells[0].find("div", class_="name")
            if name_tag:
                name = name_tag.get_text(strip=True)
            else:
                name = cells[0].get_text(strip=True)

            title = cells[1].get_text(strip=True)

            staff = {
                "school": school,
                "conference": conference,
                "department": current_dept,
                "name": name,
                "title": title
            }

            staff_list.append(staff)

    # Convert to DataFrame
    df = pd.DataFrame(staff_list)
    print(f"{school}: scraped {len(df)} staff members")

    return df


In [ ]:
"""
All staff + department headers are in <tr> tags. 
Department headers have class "subtitle" and contain a <td> 
<div class="staff-directory__department-heading"> with the dept name in <span>
The staff members for that department follow in <tr> tags.
The name is in the first <td> under <span> <a> tag.
The title is in the next <td> under a <p> tag.

Schools:
- SEC: Vanderbilt
"""

def scrape_subtitle_tr(url, school, conference="SEC"):
    import requests
    from bs4 import BeautifulSoup
    import pandas as pd

    # Fetch the page
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
    except Exception as e:
        print(f"Error fetching {school}: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(response.text, "html.parser")

    staff_list = []
    current_department = None

    # Iterate through all table rows in order
    for row in soup.find_all("tr"):

        # ------------------------
        # Department header <tr>
        # ------------------------
        if "subtitle" in row.get("class", []):
            dept_tag = row.find("div", class_="staff-directory__department-heading")
            if dept_tag:
                span = dept_tag.find("span")
                if span:
                    current_department = span.get_text(strip=True)
            continue

        # ------------------------
        # Staff member row
        # ------------------------
        cells = row.find_all("td")
        if len(cells) >= 2:  # must have name + title
            # Name
            name_tag = cells[0].find("a")
            name = name_tag.get_text(strip=True) if name_tag else None

            # Title
            title = cells[1].get_text(strip=True) if cells[1] else ""

            # Only record if it's a real staff row
            if name:
                staff_list.append({
                    "name": name,
                    "title": title,
                    "department": current_department or "",
                    "school": school,
                    "conference": conference
                })

    df = pd.DataFrame(staff_list)
    print(f"{school}: Scraped {len(df)} staff members.")
    return df


***Individual School Checks***

In [ ]:
### Alabama
school_df = scrape_div_h3_h4(url = "https://rolltide.com/staff-directory", school = "Alabama", conference = "SEC")
school_df.head(20)

In [ ]:
### Arkansas
school_df = scrape_thead_tbody_tables(url = "https://arkansasrazorbacks.com/staff-directory/", school = "Arkansas")
school_df.head(20)

In [ ]:
### Auburn
school_df = scrape_tr_td_directory_table(url = "https://auburntigers.com/staff-directory", school = "Auburn")
school_df.head(20)

In [ ]:
### Kentucky
school_df = scrape_tr_tr_table_row(url = "https://ukathletics.com/staff-directory/", school = "Kentucky")
school_df['department'].value_counts()


In [ ]:
### LSU
school_df = scrape_tr_tr_table_row(url = "https://lsusports.net/staff-directory/", school = "LSU")
school_df['department'].value_counts()

In [ ]:
### South Carolina
school_df = scrape_tr_tr_table_row(url = "https://gamecocksonline.com/staff-directory/", school = "South Carolina")
school_df['department'].value_counts()

In [ ]:
### Vanderbilt
school_df = scrape_subtitle_tr(url="https://vucommodores.com/staff-directory/", school="Vanderbilt")
school_df['department'].value_counts()

***Scrape SEC***

In [ ]:
# Whole conference scraper (change conference name)
def scrape_sec():
    # List out all schools with URL, scraper function name (elements after scrape_ in func name), and conference)
    # Duplicate the {...} for as many schools as you have and change names
    schools = [
        {'school': 'Alabama', 'url': 'https://rolltide.com/staff-directory', 'scraper': 'div_h3_h4', 'conference': 'SEC'},
        {'school': 'Arkansas', 'url': 'https://arkansasrazorbacks.com/staff-directory/', 'scraper': 'thead_tbody_tables', 'conference': 'SEC'},
        {'school': 'Auburn', 'url': 'https://auburntigers.com/staff-directory', 'scraper': 'tr_td_directory', 'conference': 'SEC'},
        {'school': 'Florida', 'url': 'https://floridagators.com/staff-directory', 'scraper': 'div_h3_h4', 'conference': 'SEC'},
        {'school': 'Georgia', 'url': 'https://georgiadogs.com/staff-directory', 'scraper': 'div_h3_h4', 'conference': 'SEC'},
        {'school': 'Kentucky', 'url': 'https://ukathletics.com/staff-directory/', 'scraper': 'tr_tr_table_row', 'conference': 'SEC'},
        {'school': 'Louisiana State', 'url': 'https://lsusports.net/staff-directory/', 'scraper': 'tr_tr_table_row', 'conference': 'SEC'},
        {'school': 'Ole Miss', 'url': 'https://olemisssports.com/staff-directory', 'scraper': 'div_h3_h4', 'conference': 'SEC'},
        {'school': 'Mississippi State', 'url': 'https://hailstate.com/staff-directory', 'scraper': 'div_h3_h4', 'conference': 'SEC'},
        {'school': 'Missouri', 'url': 'https://mutigers.com/staff-directory', 'scraper': 'div_h3_h4', 'conference': 'SEC'},
        {'school': 'Oklahoma', 'url': 'https://soonersports.com/staff-directory', 'scraper': 'div_h3_h4', 'conference': 'SEC'},
        {'school': 'South Carolina', 'url': 'https://gamecocksonline.com/staff-directory/', 'scraper': 'tr_tr_table_row', 'conference': 'SEC'},
        {'school': 'Tennessee', 'url': 'https://utsports.com/staff-directory', 'scraper': 'div_h3_h4', 'conference': 'SEC'},
        {'school': 'Texas', 'url': 'https://texaslonghorns.com/staff-directory?path=general', 'scraper': 'div_h3_h4', 'conference': 'SEC'},
        {'school': 'Texas A&M', 'url': 'https://12thman.com/staff-directory', 'scraper': 'div_h3_h4', 'conference': 'SEC'},
        {'school': 'Vanderbilt', 'url': 'https://vucommodores.com/staff-directory/', 'scraper': 'subtitle_tr', 'conference': 'SEC'}
    ]    
    
    # Scrape all schools
    # Add other scrapers if necessary here
    all_staff = []
    for s in schools:
        if s['scraper'] == 'thead_tbody_tables':
            df = scrape_thead_tbody_tables(url = s['url'], school = s['school'], conference = s['conference'])
        elif s['scraper'] == 'tr_tr_table_row':
            df = scrape_tr_tr_table_row(url = s['url'], school = s['school'], conference = s['conference'])
        elif s['scraper'] == 'div_h3_h4':
            df = scrape_div_h3_h4(url = s['url'], school = s['school'], conference = s['conference'])
        elif s['scraper'] == 'tr_td_directory':
            df = scrape_tr_td_directory_table(url = s['url'], school = s['school'], conference = s['conference'])
        elif s['scraper'] == 'subtitle_tr':
            df = scrape_subtitle_tr(url = s['url'], school = s['school'], conference = s['conference'])
            
        all_staff.append(df)
        
    ### Combine all dataframes, return it
    combined_df = pd.concat(all_staff, ignore_index=True)
    return combined_df

In [ ]:
# Run the conference scraper, add to a csv
sec_df = scrape_sec()
sec_df.to_csv('sec_staff_directory.csv', index=False)